# OpenFOAM RL Environment Demo

Tests `OpenFOAMSimulator` and `OpenFOAMEnv` using a real OpenFOAM simulation (`scalarTransportFoam`).

In [1]:
%%capture
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
import xarray as xr
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import uqtopus as uqt

## 1. OpenFOAM Simulator Setup

In [3]:
sim = uqt.OpenFOAMSimulator(
    template_path="templates/scalarTransportFoam",
    solver_script="Allrun",
    output_path="experiments/rl_runs",
    qoi_variables=["T"],
    qoi_times=["0.1"]
)
sim

OpenFOAMSimulator(template='scalarTransportFoam', solver='Allrun', runs=0)

Run a single simulation to verify OpenFOAM runs and returns results

In [4]:
PARAM_KEY = "constant__transportProperties__DT"
ds = sim.run({PARAM_KEY: 0.01}, verbose=True)
ds

Created parent directory: experiments/rl_runs
sending incremental file list
./
Allrun
0/
0/T
0/U
constant/
constant/transportProperties
system/
system/controlDict
system/fvSchemes
system/fvSolution

sent 344,134 bytes  received 172 bytes  688,612.00 bytes/sec
total size is 343,462  speedup is 1.00



PermissionError: [Errno 13] Permission denied: './Allrun'

## 2. Gym Environment Setup

In [ ]:
from uqtopus.envs import OpenFOAMEnv

TARGET_T = 0.5

def obs_fn(ds):
    T_val = ds["T"].values[0]
    return np.array([float(T_val.mean()), float(T_val.max())], dtype=np.float32)

def reward_fn(ds):
    max_T = float(ds["T"].values[0].max())
    return -abs(max_T - TARGET_T)

sim.reset()

env = OpenFOAMEnv(
    simulator=sim,
    param_ranges={PARAM_KEY: (0.001, 0.1)},
    observation_fn=obs_fn,
    reward_fn=reward_fn,
    obs_shape=(2,),
    max_episode_steps=5,
    initial_params={PARAM_KEY: 0.01},
)

print(env)
print("action_space:", env.action_space)
print("observation_space:", env.observation_space)

## 3. Manual Episode Run

In [ ]:
obs, info = env.reset()
print(f"reset  obs={obs}  params={info['params']}")

for step in range(3):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    dt_val = info["params"][PARAM_KEY]
    print(f"step {step + 1}  DT={dt_val:.2e}  reward={reward:.4f}  obs={obs}")
    if terminated or truncated:
        break

## 4. Training with stable-baselines3

In [ ]:
from stable_baselines3 import PPO
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=100)